# Step 8.5: Fine-tune the verifier (Colab A100)

Transfer learning: start from the DeBERTa-v3 NLI checkpoints the engine already uses and fine-tune on scientific / health fact-checking data (SciFact train, HealthVer, COVID-Fact).

**Runtime → Change runtime type → A100 GPU**, then run the cells in order (~45-60 min total).

Never used for training or model selection: SciFact dev (our benchmark) and HealthVer test.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/Anthony-Abumanneh/ai-truth-engine.git
%cd ai-truth-engine
!pip install -q -r requirements-train.txt

## 1. Data and starting models

In [ ]:
!mkdir -p data/scifact && curl -sL https://scifact.s3-us-west-2.amazonaws.com/release/latest/data.tar.gz | tar -xz -C data/scifact
!python -m truth_engine.pipeline.nli download large
!python -m truth_engine.pipeline.nli download base
!python -m truth_engine.bench.training_data

## 2. Before: the pretrained models

In [ ]:
!TE_NLI=1 python scripts/eval_nli.py large base

## 3. Fine-tune (large ~25 min, base ~8 min on an A100)

In [ ]:
!python scripts/finetune_verifier.py --size large

In [ ]:
!python scripts/finetune_verifier.py --size base

## 4. After: same evaluation

In [ ]:
!TE_NLI=1 python scripts/eval_nli.py large ft-large base ft-base
!TE_NLI=1 TE_NLI_MODEL=ft-large python scripts/eval_stance_scifact.py dev

## 5. Save to Google Drive

Download `truth_engine_finetuned.zip` from Drive and unzip it inside the project folder on your Mac (it contains `data/models/local__...` and the eval results).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!zip -qr truth_engine_finetuned.zip data/models/local__truth-engine-verifier-large data/models/local__truth-engine-verifier-base data/finetune/eval_nli.json data/finetune/stats.json
!cp truth_engine_finetuned.zip /content/drive/MyDrive/ && ls -lh /content/drive/MyDrive/truth_engine_finetuned.zip